# 05 - Model Evaluation

This notebook loads the test set and the two trained models, evaluates each on accuracy, precision, recall, and F1, plots and saves confusion matrices, and saves the better-performing model as `../models/best_model.pkl` for the Streamlit app.

## 1. Imports and load data

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

sns.set_theme(style='whitegrid')

X_test = pd.read_pickle('../data/X_test.pkl')
y_test = pd.read_pickle('../data/y_test.pkl')

os.makedirs('../results', exist_ok=True)

log_reg = joblib.load('../models/logistic_regression.pkl')
xgb_clf = joblib.load('../models/xgboost_model.pkl')

models = {
    'Logistic Regression': log_reg,
    'XGBoost': xgb_clf,
}

## 2. Evaluation helper

A small helper that computes all metrics for a model, prints a classification report, and plots + saves a confusion matrix PNG.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f'=== {name} ===')
    print(f'Accuracy:  {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall:    {rec:.4f}')
    print(f'F1 Score:  {f1:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()

    fname = name.lower().replace(' ', '_')
    out_path = f'../results/confusion_matrix_{fname}.png'
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'Saved confusion matrix to {out_path}')
    print()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

## 3. Evaluate both models

In [ ]:
results = {}
for name, model in models.items():
    results[name] = evaluate_model(name, model, X_test, y_test)

## 4. Side-by-side comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('f1', ascending=False)
results_df

## 5. Select and save the best model

We pick the model with the highest F1 score (a balanced measure of precision and recall, which matters here since both false positives — wasted retention offers — and false negatives — missed churners — carry a cost) and save it as `../models/best_model.pkl` for the Streamlit app to load.

In [ ]:
best_model_name = results_df.index[0]
best_model = models[best_model_name]

joblib.dump(best_model, '../models/best_model.pkl')
print(f'Best model: {best_model_name} (F1 = {results_df.loc[best_model_name, "f1"]:.4f})')
print('Saved to ../models/best_model.pkl')

## Model Comparison & Recommendation

Both models were trained on the SMOTE-resampled training set and evaluated on the original, untouched test set.

- **Logistic Regression** is fast, simple, and interpretable, and serves as a solid baseline, but as a linear model it can struggle to capture non-linear interactions between features such as tenure, contract type, and monthly charges.
- **XGBoost** is a gradient-boosted ensemble that can model those non-linear interactions, and typically edges out the baseline on recall/F1 for this kind of tabular problem, at some cost to interpretability.

The table above ranks both models by F1 score. The higher-ranked model has been saved to `../models/best_model.pkl` and is the one the Streamlit app (`app/streamlit_app.py`) loads for live predictions. If you retrain on different data and the ranking changes, simply re-run this notebook — it always saves whichever model actually scored highest as `best_model.pkl`, so the app stays in sync automatically.